# Natural Language Inference

**Adam Ek, Bill Noble, Simon Dobnik, and others**

The lab is an exploration and learning exercise to be done in a group and also in discussion with the teachers and other students.

Before starting, please read the instructions on how to work in groups on Canvas.

Write all your answers and the code in the appropriate boxes below.

In this lab we will work with neural networks for natural language inference. Our task is: given a premise sentence P and hypothesis H, what entailment relationship holds between them? Is H entailed by P, contradicted by P or neutral towards P?

Given a sentence P, if H definitely describe something true given P then it is an **entailment**. If H describe something that's *maybe* true given P, it's **neutral**, and if H describe something that's definitely *false* given P it's a **contradiction**. 

**Dependencies**

* Pytorch
    * Installation instructions: https://pytorch.org/
    * Tutorials: https://pytorch.org/tutorials/beginner/basics/intro.html
    * Some useful basic operations: https://jhui.github.io/2018/02/09/PyTorch-Basic-operations
* ...

**Running the code**

As we are learning about the models, and also what methods work and do not work for our semantic tasks, we are not interested in achieving a state-of-the-art performance. We are learning about different implementations and differences in performance in different conditions.

**On using generative AI for this assignment:** For this lab, the use of generative AI is permitted as a supporting tool, provided it is done in a responsible and conscious manner and that you state clearly with each question how it was used. However, generative AI must never replace the intellectual work you are expected to carry out. Note that the purpose of this lab is to learn some basic coding of the main neural architectures used in natural language processing. You are responsible for ensuring that such tools are used in a way that supports the development of the skills the module is designed to promote. It is your responsibility to ensure that submitted work is the result of independent intellectual effort.

**Getting help:** We encourage you to use Canvas discussions to post questions and interact with teachers and also each other. Provide useful tips, but of course do not reveal the exact answer across the groups as each group should should work out their own solutions. Remember that in most cases there is also not a single correct answer and implementations may differ.

## 1. Data

We will explore natural language inference using neural networks on the SNLI dataset, described in [1]. 

There are two options for loading and working with the data.

1. Download the data directly from the [SNLI website](https://nlp.stanford.edu/projects/snli/) and write a dataloader based on your dataloader from **A3: Distributed Representations and Language Models**.
2. Use the `datasets` library to load the version on the [HuggingFace hub](https://huggingface.co/datasets/stanfordnlp/snli). Follow the steps in [the documentation](https://huggingface.co/docs/datasets/v2.19.0/loading#hugging-face-hub) for loading the dataset.

[you can remove the template for whatever code you don't use]

The data is organized as follows:

* Column 1: Premise (sentence1)
* Column 2: Hypothesis (sentence2)
* Column 3: Relation (gold_label)

**[3 marks]**

In [ ]:
%pip install -r packages.txt

In [ ]:
import datasets
from datasets import load_dataset

# enable progress bars
datasets.enable_progress_bar()

# set verbose logging
datasets.logging.set_verbosity_info()

#Loading SNLI dataset and each file is tab -separated and has 3 columns: premise, hypothesis, label
dataset = load_dataset("csv", data_files={
    "train": "./simple_snli_1.0/simple_snli_1.0_train.csv",             #training split
    "validation": "./simple_snli_1.0/simple_snli_1.0_dev.csv",          #validation split
    "test": "./simple_snli_1.0/simple_snli_1.0_test.csv"                #test split
}, delimiter="\t", column_names=["premise", "hypothesis", "label"])     #tab- separated, 3 columns

In [ ]:
# print the first 10 items of the training dataset
for i in range(10):
    print(f"Item {i}:")
    print(f"Premise:    {dataset['train'][i]['premise']}")
    print(f"Hypothesis: {dataset['train'][i]['hypothesis']}")
    print(f"Label:      {dataset['train'][i]['label']}")
    print("-" * 50)

Notice that the dataset comes as a dictionary-like object with three splits: `'test'`, `'train'`, and `'validation'`. Each item is a dictionary containing a `'premise'`, `'hypothesis'`, and `'label'`.

## 2. Tokenization

This data does not come pre-tokenized. Instead of training our own tokenizer, we can use the BERT tokenizer like in the preivous assignment. Even though we aren't using BERT the tokenizer works with any model. See the documentation on [using a pretrained tokenizer](https://huggingface.co/docs/tokenizers/en/quicktour#using-a-pretrained-tokenizer). **[1 mark]**

In [ ]:
from tokenizers import Tokenizer
#loading pre-trained BERT tokenizer
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")

#test the tokenizer on the first training example
ex = dataset['train'][0]
print("Example Premise:", ex['premise'])                #print raw sentence
print("Encoded IDs:", tokenizer.encode(ex['premise']).ids)          #print token ids

## 2. Model

In this part, we'll build the model for predicting the relationship between H and P.

We will process each sentence using an LSTM. Then, we will construct some representation of the sentence. When we have a representation for H and P, we will combine them into one vector which we can use to predict the relationship.

We will train a model described in [2], the BiLSTM with max-pooling model. The procedure for the model is roughly:

    1) Encode the Hypothesis and the Premise using one shared bidirectional LSTM (or two different LSTMS)
    2) Perform max over the tokens in the premise and the hypothesis
    3) Combine the encoded premise and encoded hypothesis into one representation
    4) Predict the relationship 

### Creating a representation of a sentence

Let's first consider step 2 where we perform pooling. There is a builtin function in pytorch for this, but we'll implement it from scratch. 

Let's consider the general case, what we want to do for these methods is apply some function $f$ along dimension $i$, and we want to do this for all $i$'s. As an example we consider the matrix S with size ``(N, D)`` where N is the number of words and D the number of dimensions:

$S = \begin{bmatrix}
    s_{11} & s_{12} & s_{13} & \dots  & s_{1d} \\
    s_{21} & s_{22} & s_{23} & \dots  & s_{2d} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    s_{n1} & s_{n2} & s_{n3} & \dots  & s_{nd}
\end{bmatrix}$

What we want to do is apply our function $f$ on each dimension, taking the input $s_{1d}, s_{2d}, ..., s_{nd}$ and generating the output $x_d$. 

You will implement the max pooling method. When performing max-pooling, $max$ will be the function which selects a _maximum_ value from a vector and $x$ is the output, thus for each dimension $d$ in our output $x$ we get:

\begin{equation}
    x_d = max(s_{1d}, s_{2d}, ..., s_{nd})
\end{equation}

This operation will reduce a batch of size ``(batch_size, num_words, dimensions)`` to ``(batch_size, dimensions)`` meaning that we now have created a sentence representation based on the content of the representation at each token position. 

Create a function that takes as input a tensor of size ``(batch_size, num_words, dimensions)`` then performs max pooling and returns the result (the output should be of size: ```(batch_size, dimensions)```). [**4 Marks**]

In [ ]:
import torch

def max_pooling(input_tensor):
    # input_tensor has shape: (batch_size, num_words, dimensions)
    # we take the maximum value along the words dimension (dim=1)
    # torch.max returns a tuple of (values, indices), we only need the values
    output_tensor, _ = torch.max(input_tensor, dim=1)
    return output_tensor            #shape: (batch_size, dimensions)

test_unpooled = torch.rand(32, 100, 512)
test_pooled = max_pooling(test_unpooled)
print(test_pooled.size()) # should be torch.Size([32, 512])

output:

torch.Size([32, 512])

### Combining sentence representations

Next, we need to combine the premise and hypothesis into one representation. We will do this by concatenating four tensors (the final size of our tensor $X$ should be ``(batch_size, 4d)`` where ``d`` is the number of dimensions that you use): 

$$X = [P; H; |P-H|; P \cdot H]$$

Here, what we do is concatenating P, H, P times H, and the absolute value of P minus H, then return the result.

Implement the function. **[4 marks]**

In [ ]:
def combine_premise_and_hypothesis(hypothesis, premise):
    # P times H
    product = premise * hypothesis

    # the absolute value of P minus H
    abs_diff = torch.abs(premise - hypothesis)

    #concatenating all 4 vectors along the feature dimension
    output = torch.cat((hypothesis, premise, product, abs_diff), dim=1)
    return output

#sanity check
test_hypothesis = test_pooled.clone()
test_premise = test_pooled.clone()

#combines them - 
test_combined = combine_premise_and_hypothesis(test_hypothesis, test_premise)
assert test_combined.size() == torch.Size([32, 2048]), f"Unexpected shape: {test_combined.size()}"
print(test_combined.size())  # should be torch.Size([32, 400])

Output:

torch.Size([32, 2048])

### Creating the model

Finally, we can build the model according to the procedure given previously by using the functions we defined above. Additionaly, in the model you should use *dropout*. For efficiency purposes, it's acceptable to only train the model with either max or mean pooling. 

Implement the model [**8 marks**]

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# model hyperparameters
EMBEDDING_DIM = 128
HIDDEN_SIZE = 128
NUM_CLASSES = 3
DROPOUT_RATE = 0.1


class SNLIModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_size=HIDDEN_SIZE,
        num_classes=NUM_CLASSES,
        dropout_rate=DROPOUT_RATE
    ):
        super().__init__()

        # embed token ids to dense vectors
        self.embeddings = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0)

        # shared bidirectional LSTM encoder
        self.rnn = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            bidirectional=True,
            batch_first=True
        )

        # dropout to prevent overfitting
        self.dropout = nn.Dropout(dropout_rate)

        # linear classifier projecting combined representation to output classes
        # hidden_size * 2 because the LSTM is bidirectional
        # multiplied by 4 because we concatenate 4 vectors: P, H, |P-H|, and P*H
        self.classifier = nn.Linear(4 * (hidden_size * 2), num_classes)

    def _encode(self, token_ids):
        lengths = (token_ids != 0).sum(dim=1).clamp(min=1).cpu()
        #convert token IDs to emb vectors
        emb = self.embeddings(token_ids)
        #LSTM skips padding position
        packed = pack_padded_sequence(emb, lengths,
                                      batch_first=True, enforce_sorted=False)
        out_packed, _=self.rnn(packed)
        out, _= pad_packed_sequence(out_packed, batch_first=True)

        #zero out padding positions before pooling so they cannot win the max
        mask = (token_ids != 0).unsqueeze(-1).float()
        out = out * mask
        return max_pooling(out)
        

    def forward(self, premise, hypothesis):

        # max pool over the words of both encoded sentences
        p_pooled = self._encode(premise)
        h_pooled = self._encode(hypothesis)

        # combine into one representations
        combined = combine_premise_and_hypothesis(p_pooled, h_pooled)
        combined = self.dropout(combined)
        predictions = self.classifier(combined)

        return predictions

## 3. Training

As before, implement the training and testing of the model. SNLI can take a very long time to train, so I suggest you only run it for one or two epochs. **[10 marks]** 

**Tip for efficiency:** *when developing your model, try training and testing the model on one batch (for each epoch) of data to make sure everything works! It's very annoying if you train for N epochs to find out that something went wrong when testing the model, or to find that something goes wrong when moving from epoch 0 to epoch 1.*

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

def tokenize_and_pad(sentences):
    # tokenize each sentence and pad to the same length within the batch
    encoded = [torch.tensor(tokenizer.encode(s).ids) for s in sentences]
    return pad_sequence(encoded, batch_first=True, padding_value=0)

def prepare_batch(batch, label_map, device):
    """Filter invalid rows, tokenize, and move tensors to device. 
    returns (p_tensor, h_tensor, l_tensor) or none if the batch is entirely invalid."""
    valid = [(p,h,l)
            for p, h, l in zip(batch['premise'], batch['hypothesis'], batch['label'])
            if l in label_map and p is not None and h is not None
            ]
    if not valid:
        return None         #signal to skip this batch entirely
    #separate into individual lists
    premises, hypothesis, labels = zip(*valid)

    #tokenize, pad and move each tesnor to correct device (CPU or GPU)
    p_tensor = tokenize_and_pad(premises).to(device)
    h_tensor = tokenize_and_pad(hypothesis).to(device)
    l_tensor = torch.tensor([label_map[l] for l in labels], device = device)
    return p_tensor, h_tensor, l_tensor

#config
epochs = 2
batch_size = 32

#maps string labels -> integer class indices for the loss function
label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
#reverse map
label_names = {v: k for k, v in label_map.items()}

#move everything to GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

loss_function = nn.CrossEntropyLoss()
model = SNLIModel(vocab_size=len(tokenizer.get_vocab())).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=1, factor=0.5, verbose=True)

#Training loop
for epoch in range(epochs):
    model.train()
    train_iter = dataset['train'].iter(batch_size=batch_size)
    total_loss = 0
    batch_count = 0

    for batch in train_iter:
        result = prepare_batch(batch, label_map, device)
        if result is None:
            continue
        #unpack three tensors returnes by prepare_batch
        p_tensor, h_tensor, l_tensor = result

        optimizer.zero_grad()
        predictions = model(p_tensor, h_tensor)
        loss = loss_function(predictions, l_tensor)
        loss.backward()
        #gradients clipping prevents exploding gradients in LSTM
        nn.utils.clip_grad_norm_(model.parameters(), max_norm= 5.0)
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        #print average loss every 5-- batches to monitor training progress
        if batch_count % 500 == 0:
            avg = total_loss / batch_count
            print(f"Epoch {epoch+1} | Batch {batch_count} | Avg loss: {avg:.4f}")
    #compute average loss for the whole epoch
    epoch_loss= total_loss / batch_count
    print(f"Epoch {epoch+1} complete | Avg loss: {total_loss / batch_count:.4f}")
    #pass epoch loss to scheduler
    scheduler.step(epoch_loss)

print("Training complete.")

Epoch 2 | Batch 7500 | Avg loss: 0.5927
Epoch 2 | Batch 8000 | Avg loss: 0.5918
Epoch 2 | Batch 8500 | Avg loss: 0.5912
Epoch 2 | Batch 9000 | Avg loss: 0.5903
Epoch 2 | Batch 9500 | Avg loss: 0.5896
Epoch 2 | Batch 10000 | Avg loss: 0.5888
Epoch 2 | Batch 10500 | Avg loss: 0.5880
Epoch 2 | Batch 11000 | Avg loss: 0.5866
Epoch 2 | Batch 11500 | Avg loss: 0.5860
Epoch 2 | Batch 12000 | Avg loss: 0.5852
Epoch 2 | Batch 12500 | Avg loss: 0.5848
Epoch 2 | Batch 13000 | Avg loss: 0.5846
Epoch 2 | Batch 13500 | Avg loss: 0.5843
Epoch 2 | Batch 14000 | Avg loss: 0.5834
Epoch 2 | Batch 14500 | Avg loss: 0.5829
Epoch 2 | Batch 15000 | Avg loss: 0.5822
Epoch 2 | Batch 15500 | Avg loss: 0.5815
Epoch 2 | Batch 16000 | Avg loss: 0.5809
Epoch 2 | Batch 16500 | Avg loss: 0.5806
Epoch 2 | Batch 17000 | Avg loss: 0.5798
Epoch 2 complete | Avg loss: 0.5796
Training complete.


Output: 

Using device: cuda

Epoch 1 | Batch 500 | Avg loss: 0.9601
Epoch 1 | Batch 1000 | Avg loss: 0.9032
Epoch 1 | Batch 1500 | Avg loss: 0.8656
Epoch 1 | Batch 2000 | Avg loss: 0.8395
Epoch 1 | Batch 2500 | Avg loss: 0.8207
Epoch 1 | Batch 3000 | Avg loss: 0.8036
Epoch 1 | Batch 3500 | Avg loss: 0.7914
Epoch 1 | Batch 4000 | Avg loss: 0.7813
Epoch 1 | Batch 4500 | Avg loss: 0.7715
Epoch 1 | Batch 5000 | Avg loss: 0.7634
Epoch 1 | Batch 5500 | Avg loss: 0.7571
Epoch 1 | Batch 6000 | Avg loss: 0.7505
Epoch 1 | Batch 6500 | Avg loss: 0.7448
Epoch 1 | Batch 7000 | Avg loss: 0.7390
Epoch 1 | Batch 7500 | Avg loss: 0.7343
Epoch 1 | Batch 8000 | Avg loss: 0.7294
Epoch 1 | Batch 8500 | Avg loss: 0.7245
Epoch 1 | Batch 9000 | Avg loss: 0.7199
Epoch 1 | Batch 9500 | Avg loss: 0.7165
Epoch 1 | Batch 10000 | Avg loss: 0.7130
Epoch 1 | Batch 10500 | Avg loss: 0.7097
Epoch 1 | Batch 11000 | Avg loss: 0.7059
Epoch 1 | Batch 11500 | Avg loss: 0.7029
Epoch 1 | Batch 12000 | Avg loss: 0.7001
Epoch 1 | Batch 12500 | Avg loss: 0.6975
Epoch 1 | Batch 13000 | Avg loss: 0.6954
Epoch 1 | Batch 13500 | Avg loss: 0.6933
Epoch 1 | Batch 14000 | Avg loss: 0.6904
Epoch 1 | Batch 14500 | Avg loss: 0.6883
Epoch 1 | Batch 15000 | Avg loss: 0.6859
Epoch 1 | Batch 15500 | Avg loss: 0.6836
Epoch 1 | Batch 16000 | Avg loss: 0.6815
Epoch 1 | Batch 16500 | Avg loss: 0.6796
Epoch 1 | Batch 17000 | Avg loss: 0.6773
Epoch 1 complete | Avg loss: 0.6766
Epoch 2 | Batch 500 | Avg loss: 0.6116
Epoch 2 | Batch 1000 | Avg loss: 0.6099
Epoch 2 | Batch 1500 | Avg loss: 0.6075
Epoch 2 | Batch 2000 | Avg loss: 0.6026
Epoch 2 | Batch 2500 | Avg loss: 0.6014
Epoch 2 | Batch 3000 | Avg loss: 0.5976
Epoch 2 | Batch 3500 | Avg loss: 0.5981
Epoch 2 | Batch 4000 | Avg loss: 0.5976
Epoch 2 | Batch 4500 | Avg loss: 0.5958
Epoch 2 | Batch 5000 | Avg loss: 0.5949
Epoch 2 | Batch 5500 | Avg loss: 0.5948
Epoch 2 | Batch 6000 | Avg loss: 0.5937
Epoch 2 | Batch 6500 | Avg loss: 0.5931
Epoch 2 | Batch 7000 | Avg loss: 0.5921
Epoch 2 | Batch 7500 | Avg loss: 0.5913
Epoch 2 | Batch 8000 | Avg loss: 0.5904
Epoch 2 | Batch 8500 | Avg loss: 0.5892
Epoch 2 | Batch 9000 | Avg loss: 0.5882
Epoch 2 | Batch 9500 | Avg loss: 0.5876
Epoch 2 | Batch 10000 | Avg loss: 0.5865
Epoch 2 | Batch 10500 | Avg loss: 0.5856
Epoch 2 | Batch 11000 | Avg loss: 0.5844
Epoch 2 | Batch 11500 | Avg loss: 0.5838
Epoch 2 | Batch 12000 | Avg loss: 0.5830
Epoch 2 | Batch 12500 | Avg loss: 0.5824
Epoch 2 | Batch 13000 | Avg loss: 0.5822
Epoch 2 | Batch 13500 | Avg loss: 0.5820
Epoch 2 | Batch 14000 | Avg loss: 0.5812
Epoch 2 | Batch 14500 | Avg loss: 0.5807
Epoch 2 | Batch 15000 | Avg loss: 0.5801
Epoch 2 | Batch 15500 | Avg loss: 0.5793
Epoch 2 | Batch 16000 | Avg loss: 0.5787
Epoch 2 | Batch 16500 | Avg loss: 0.5781
Epoch 2 | Batch 17000 | Avg loss: 0.5774
Epoch 2 complete | Avg loss: 0.5771
Training complete.

## 4. Testing

Test the model on the testset. For each example in the test set, compute a prediction from the model (`entailment`, `contradiction` or `neutral`). Compute precision, recall, and F1 score for each label. **[10 marks]**

In [ ]:
from sklearn.metrics import classification_report
#set model to eval
model.eval()

all_preds = []
all_true = []

# run inference on the full test set without computing gradients
with torch.no_grad():
    for batch in dataset['test'].iter(batch_size=batch_size):
        result = prepare_batch(batch, label_map, device)
        if result is None:
            continue        #skip invalid batches
        #unpack tensors
        p_tensor, h_tensor, l_tensor = result

        # forward pass- get raw logits from the model
        predictions = model(p_tensor, h_tensor)
        pred_labels = torch.argmax(predictions, dim=1).tolist()

        #collect predictions and true labels for this batch
        all_preds.extend(pred_labels)
        all_true.extend(l_tensor.tolist())

print(classification_report(
    all_true,
    all_preds,
    target_names=["entailment", "neutral", "contradiction"]
))

Output: 

                precision    recall  f1-score   support

   entailment       0.78      0.86      0.82      3368
      neutral       0.76      0.69      0.72      3219
contradiction       0.81      0.80      0.80      3237

     accuracy                           0.78      9824
    macro avg       0.78      0.78      0.78      9824
 weighted avg       0.78      0.78      0.78      9824

Suggest a _baseline_ that we can compare our model against **[2 marks]**

**Your answer should go here**

A strong and simple baseline is a majority class baseline, where we always predict the most frequent label in the training data. Since SNLI is roughly balanced across entailemnt, contradiction, and neutral, this gives about ~33% accuracy, which serves as a lower bound that any trained model should outperform.
A slightly stronger baseline is a bag-of-words(BOW) approach combined with logistic regression. In this method we represent the premise and hypothesis using averaged word embeddings (such as Glove or even random embeddings), concatenate them, and trian a logistic regression classifier on top. This baseline is fast to train and typically achieves around 65 - 70% accuracy on SNLI, providing a good comparision point to show that more advanced models like a BiLSTM with max-pooling (~78%) are learning deeper sentence structure rather than just relying on surface-level word overlap. Our BiLSTM with max-pooling achieved 78% accuracy, outperforming both baselines and confirming that the model is learning deeper linguistic structure beyond surface-level word overlap.


Suggest some ways (other than using a baseline) in which we can analyse the models performance **[3 marks]**.

**Your answer should go here**

We can analyse model performance in few ways beyond just overall accuracy or baseline comparision

One approach is per-class error analysis, where we look at the confusion matrix to understnad which labels are most often confused with each other, such as neutral vs entailement. We can also manually inspect incorrectly predicted examples to detect patterns in the errors, for example whether the models struggels with negation, numerical reasoning or longer and more complex sentences.For example, in our results the neutral class had the lowest recall(0.69), suggesting the model most often confuses neutral with entailment- a pattern that manual error analysis could help explain.

Another useful method is sentences length analysis, where test examples are grouped into buckets based on the length of the premise or hypothesis. We then measure accuracy within each bucket to see whether performance drops for longer inputs, which is a common issue for models that rely on fixed- size sentence representaions.

Fianlly, we can do stress testing on challenging or adversarial examples, such as datasets designed to test negation, antonyms, or world knowldege. This helps us understand whether the model is truly learning inference or simply relying on shallow cues like word overlap.


Suggest some ways to improve the model **[3 marks]**.

**Your answer should go here**

One way to improve the model is to use pre-trained word embeddings like Glove or fastText instead of randomnly initialised embeddings. These embeddings already capture rich semantic realtionships between words, which gives the model a much better starting point and usually improves performance on SNLI by a few percentage points.

Another important is to include an attention mechanism instead of relying only on max-pooling. For example, a cross-attention layer can allow the hypothesis to attend to relevant parts of the premise (and vice versa), helping the model focus on important word alignments rather than compressing the entire sentence into a single fixed vector.

We can also improve performance by using a stronger classifier and better training strategy. Instead of a single linear layer, a small multi-layer perceptron (MLP)with a non-lineae activation (like ReLU) and dropout can capture more complex decision boundares. In addition,training for more epochs and using learning rate scheduling for eg:, reducing the learning rate when validation performance stops improving can help the model converge better.Our optimized implementation already includes ReduceLROnPlacteau scheduling and gradient clipping, which stabilizes LSTM training and prevents loss spikes.

Finally a more advanced improvement is to fine-tune a pretrianed transformer model like BERT on SNLI, which already encodes deep contextual knowledge and typically achieves much higher accuracy compared to LSTM-based models.



## Readings

[1] Samuel R. Bowman, Gabor Angeli, Christopher Potts, and Christopher D. Manning. 2015. A large annotated corpus for learning natural language inference. In Proceedings of the 2015 Conference on Empirical Methods in Natural Language Processing (EMNLP). 

[2] Conneau, A., Kiela, D., Schwenk, H., Barrault, L., & Bordes, A. (2017). Supervised learning of universal sentence representations from natural language inference data. arXiv preprint arXiv:1705.02364.

## Your reflections on this lab

Write below your general thoughts, experiences, or reflections on how you worked on this lab.

working on this lab helped us understnad how NLI models actually work in practice, not just in theory. It was intresting to see how even a relatively simple model like a  BiLSTM can already give decent results, but still has clear limitations compared to more advanced pretrianed models like BERT.

We also got a better idea of why baselines are important. At first it felt like just an extra step, but comparing against a simple baseline made it much cleare. whether the model was actullay learning something useful. Doing error analysis was also quite helpful because it showed that the model sometimes depends on simple pstterns like word overlap and can struggle with things like negation or more complex sentences.

Overall the lab was useful for getting hands on experience with training and evaluating NLP models, and it made us think more about what the model is actually learning rather than just looking at accuracy numbers.


## Statement of contribution

Briefly state how many times you have met for discussions, who was present, to what degree each member contributed to the discussion and the final answers you are submitting.

## Marks

The assignment has 45 marks.